# 06 — Generators and Lazy Pipelines

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/06', {1: 'b418da0ef76a8b7b', 2: 'f6eb54636c841643', 3: 'a4d521aed9dd88f3', 4: '1848e666799bfd53', 5: '0763c6f7b2d0cff7', 6: 'cac3ca044f622053', 7: '6e7e55c6f53dccc0', 8: 'fa83c28e7518bf88', 9: 'f9bc2da5d300b5e6'}, practice='intermediate_02_iteration_and_abstractions')

## 1. yield

- A function with `yield` is a generator: calling it returns an iterator, and its body runs a step at a time as you ask for values.
- Each `yield` hands out one value and pauses; the next `next()` resumes right after it.
- Generators are lazy: they can describe endless sequences and use almost no memory.

In [ ]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

print(list(fibonacci(8)))

**Exercise 1.** Write a generator that yields the squares of `1..n`.

In [ ]:
def squares(n):
    for i in range(1, n + 1):
        yield ___

check(1, list(squares(4)))

**Exercise 2.** Get the first six Fibonacci numbers from the generator above.

In [ ]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = ___, ___

check(2, list(fibonacci(6)))

**Exercise 3.** What does calling a generator function return before you loop? Write its type name.

In [ ]:
def gen():
    yield 1

type_name = ___
check(3, type_name)

## 2. Filtering and yield from

- Skip bad input inside the generator: `try` the conversion and `continue` on `ValueError`.
- `yield from inner` yields every item of `inner`, handy for flattening.
- Clean text lazily: `for line in lines: line = line.strip(); if line: yield line`.

In [ ]:
def positive_ints(lines):
    for line in lines:
        try:
            n = int(line)
        except ValueError:
            continue
        if n > 0:
            yield n

print(list(positive_ints(["3", "x", "-1", "8"])))

**Exercise 4.** Yield only the lines that aren't blank, stripped.

In [ ]:
def non_blank(lines):
    for line in lines:
        line = line.strip()
        if ___:
            yield line

check(4, list(non_blank(["  a ", "", "   ", "b"])))

**Exercise 5.** Flatten one level of nesting with `yield from`.

In [ ]:
def flatten(groups):
    for group in groups:
        yield from ___

check(5, list(flatten([[1, 2], [], [3]])))

**Exercise 6.** Yield only valid positive integers, skipping text that isn't a number.

In [ ]:
def positive_ints(lines):
    for line in lines:
        try:
            n = int(line)
        except ValueError:
            ___
        if n > 0:
            yield n

check(6, list(positive_ints(["5", "five", "0", "12"])))

## 3. Pipelines

- Chain generators: each stage takes an iterator and yields to the next. Nothing runs until the end is consumed.
- `itertools.accumulate(values)` yields running totals.
- Group an endless stream into chunks by taking `islice(it, size)` repeatedly.

In [ ]:
from itertools import accumulate, islice
lines = ["3", "x", "4", "5"]
numbers = (int(s) for s in lines if s.isdigit())
print(list(accumulate(numbers)))
it = iter(range(7))
print([chunk for chunk in iter(lambda: list(islice(it, 3)), [])])

**Exercise 7.** Running totals of the values with `accumulate`.

In [ ]:
from itertools import accumulate
values = [5, 1, 4]
totals = list(___)
check(7, totals)

**Exercise 8.** Build a pipeline: strip each line, keep the digits only, convert to int, then double.

In [ ]:
lines = [" 4", "a", "10 ", " "]
stripped = (s.strip() for s in lines)
numbers = (int(s) for s in stripped if ___)
doubled = [n * 2 for n in numbers]
check(8, doubled)

**Exercise 9.** `chunked` takes `size` items at a time until `islice` returns nothing. Predict the chunks of `range(7)` in 3s.

In [ ]:
from itertools import islice

def chunked(items, size):
    it = iter(items)
    while chunk := list(islice(it, size)):
        yield chunk

predicted = ___   # list(chunked(range(7), 3))
check(9, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `fibonacci`

Yield the first `n` Fibonacci numbers.

In [ ]:
def fibonacci(n: int) -> Iterator[int]:
    """Yield the first n Fibonacci numbers: 0, 1, 1, 2, 3, 5, ..."""
    a, b = ___
    for _ in ___:
        yield a
        a, b = b, a + b

In [ ]:
check('fibonacci')

### 2. `positive_ints`

Lazily yield positive integers from text, skipping invalid lines.

In [ ]:
def positive_ints(lines: Iterable[str]) -> Iterator[int]:
    """Lazily yield positive integers from text lines, skipping anything invalid."""
    for line in lines:
        try:
            value = ___
        except ValueError:
            continue
        if ___:
            yield value

In [ ]:
check('positive_ints')

### 3. `flatten_one_level`

Yield every item of every group with `yield from`.

In [ ]:
def flatten_one_level(groups: Iterable[Iterable]) -> Iterator:
    """Yield every item of every group. Hint: yield from."""
    for group in ___:
        yield from group

In [ ]:
check('flatten_one_level')

### 4. `non_blank`

Yield stripped lines, skipping blank ones.

In [ ]:
def non_blank(lines: Iterable[str]) -> Iterator[str]:
    """Lazily yield lines with surrounding whitespace removed, skipping blank ones."""
    return ___

In [ ]:
check('non_blank')

### Independent build — `chunked`

Order Desk now streams scan events. Yield lists of size items in order, including a shorter final batch. Assume size is positive. Empty input yields nothing. The first batch must be available even for an endless stream.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def chunked(iterable: Iterable, size: int) -> Iterator[list]:
    raise NotImplementedError

In [ ]:
check('chunked')

### 6. `running_total`

Yield the running total after each value.

In [ ]:
def running_total(values: Iterable[float]) -> Iterator[float]:
    """Yield the cumulative sum after each value. Hint: itertools.accumulate."""
    return ___

In [ ]:
check('running_total')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Calling a generator function…

   - **a)** Returns a generator object without running the body yet
   - **b)** Runs the whole body
   - **c)** Returns a list

**2.** `yield from sub` is equivalent to…

   - **a)** return sub
   - **b)** list(sub)
   - **c)** for x in sub: yield x

**3.** A key benefit of chaining generators is…

   - **a)** Reusability without limit
   - **b)** Random access
   - **c)** Each item flows through without building big intermediate lists

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/06', {1: '6716260bc536', 2: '2c88104cfc56', 3: '68424c39face'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [07 — Closures and Decorators](07_closures_and_decorators.ipynb)